# Notebook 11 — Duration and reported sex/gender confounder validation

This notebook evaluates whether unequal recording duration and the cohort's reported sex/gender composition can account for the participant-level EEG classification and selected HCFIM geometry results.

It is a confirmatory sensitivity analysis for the manuscript *Information-Theoretic HCFIM Geometry for Data-Limited EEG Screening of Alzheimer's Disease and Frontotemporal Dementia*.


## tl;dr

The extended notebook executed successfully for all 88 participants. It includes repeated 10×5 participant-level validation, full fixed60 confounder passthrough, two selected-geometry nuisance models with 4,999 Freedman–Lane permutations, and an FTD-control strict common-support sensitivity.

- **Control versus AD:** the fixed60 age+gender+epoch-count baseline had mean ROC AUC 0.675. Adding EEG increased mean AUC to 0.724–0.760 across the five feature families (paired mean gains 0.049–0.084). Enhanced PSD was highest (0.760); adding HCFIM did not improve it. Temporal alpha FIM remained displaced after full adjustment in both duration scopes (BH q=0.0108).
- **Control versus FTD:** the age+gender+epoch-count baseline reached mean AUC 0.872. Adding fixed60 EEG did not demonstrate incremental value; mean AUC was 0.633–0.749 and all paired changes were negative. After adding epoch count to the geometry nuisance model, temporal H, C, and FIM no longer survived BH correction (fixed60 q=0.251, 0.525, and 0.140).
- Candidate matching of all 23 FTD participants showed inadequate overlap. A strict standardized-distance caliper of 1.0 retained only 10 pairs but reduced absolute SMDs to 0.239 for age, 0 for gender, and 0.069 for epoch count. In this small subset, PSD was near chance (AUC 0.513), other EEG families were lower, and none of the paired H/C/FIM differences survived BH correction (all q=0.465).

The extended analysis supports a duration- and covariate-robust AD temporal-FIM shift, but it does **not** support general HCFIM classifier complementarity. The FTD findings are especially sensitive to cohort structure and must be presented as exploratory and confounded rather than as diagnosis-specific evidence.


## Context & Methods

### Questions

1. Does equalizing analyzed duration to the first 60 retained epochs materially change PSD, HCFIM, or combined-model performance?
2. Can epoch count alone discriminate the diagnostic groups?
3. Can age and reported sex/gender alone discriminate the groups, and do these covariates materially change EEG performance?
4. Do the selected temporal HCFIM coordinates retain a diagnosis-associated shift after adjustment for age and reported sex/gender?

### Key assumptions

- `gender` is used exactly as released in the dataset (`F`/`M`). It is described as **reported sex/gender** because the source metadata do not support a more specific construct.
- The fixed-duration matrix is not copied from the full-duration subject aggregates. It is rebuilt from `epoch_features_full.csv` using epochs 0–59 and the same six summaries: mean, standard deviation, median, 25th percentile, 75th percentile, and IQR.
- The full and fixed-duration comparisons use identical participant ordering and repeated folds.
- Age and gender are forced passthrough covariates: they are imputed/scaled within each training fold and are never allowed to compete with EEG variables in `SelectKBest`.
- All analyses remain exploratory because the cohort is small, single-source, and lacks independent external validation.

- The extended analysis also forces epoch count through fixed-duration models, adds epoch count to the selected-geometry nuisance model, and performs an exact-gender FTD-control common-support sensitivity matched on age and epoch count.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import platform
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy import signal, stats
from scipy.io import loadmat
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 17536
N_FIXED_EPOCHS = 60
N_REPEATS = 10
OUTER_FOLDS = 5
K_VALUES = [21, 30]
PRIMARY_K = 30
N_TREES = 200
MAX_DEPTH = 3
N_GEOMETRY_PERMUTATIONS = 4999
N_MATCHED_PERMUTATIONS = 9999
COMMON_SUPPORT_CALIPER = 1.0
EPOCH_CHUNK_SIZE = 500_000

QUICK_MODE = os.environ.get("HCFIM_QUICK_MODE", "0") == "1"
if QUICK_MODE:
    N_REPEATS = 2
    OUTER_FOLDS = 3
    N_TREES = 50
    N_GEOMETRY_PERMUTATIONS = 99
    N_MATCHED_PERMUTATIONS = 199

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
RESULTS = PROJECT_ROOT / "results"
INPUT_FEATURES = RESULTS / "subject_features_full.csv"
EPOCH_FEATURES = RESULTS / "epoch_features_full.csv"
PARTICIPANTS_INDEX = RESULTS / "participants_index.csv"
EPOCH_COUNTS_SOURCE = RESULTS / "reviewer_revision" / "epoch_counts_by_subject.csv"
FULL_SPECTRAL_CACHE = RESULTS / "repeated_primary_validation" / "spectral_plus_features.csv"

OUT = RESULTS / "duration_sex_confounder_validation"
FIGURES = OUT / "figures"
FIXED_FEATURE_CACHE = OUT / "subject_features_fixed60.csv"
FIXED_SPECTRAL_CACHE = OUT / "spectral_plus_fixed60.csv"
FIXED_SPECTRAL_QC_CACHE = OUT / "spectral_plus_fixed60_qc.csv"
FULL_SPECTRAL_QC_CACHE = OUT / "spectral_plus_full_qc.csv"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

FORCE_FIXED_REBUILD = os.environ.get("HCFIM_FORCE_FIXED60_REBUILD", "0") == "1"
FORCE_SPECTRAL_REBUILD = os.environ.get("HCFIM_FORCE_SPECTRAL_REBUILD", "0") == "1"

plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
})

PALETTE = {
    "full": "#315B7D",
    "fixed60": "#D18B32",
    "none": "#315B7D",
    "age_gender": "#8A5A9E",
    "confounder": "#6B7280",
}

print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Output: {OUT}")
print(f"Quick mode: {QUICK_MODE}")


## Data

### 1. Load participant, epoch-count, and full-duration feature sources

The controlling sources are the same files used by the manuscript analyses. The 3.36 GB epoch-level table is read in chunks only when the fixed-duration cache must be rebuilt.


In [ ]:
required_paths = [INPUT_FEATURES, EPOCH_FEATURES, PARTICIPANTS_INDEX, EPOCH_COUNTS_SOURCE]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

subject_features_full = pd.read_csv(INPUT_FEATURES)
participants = pd.read_csv(PARTICIPANTS_INDEX)
epoch_counts = pd.read_csv(EPOCH_COUNTS_SOURCE)

participants["participant_id"] = participants["participant_id"].astype(str)
participants["gender"] = participants["gender"].astype(str).str.strip().str.upper()
participants["age"] = pd.to_numeric(participants["age"], errors="coerce")
participants["set_exists"] = participants["set_exists"].astype(str).str.lower().eq("true")

assert len(participants) == 88 and participants["participant_id"].is_unique
assert len(subject_features_full) == 88 and subject_features_full["participant_id"].is_unique
assert set(participants["gender"]) == {"F", "M"}
assert participants["age"].notna().all()
assert participants["set_exists"].all()
assert set(subject_features_full["participant_id"]) == set(participants["participant_id"])
assert set(epoch_counts["participant_id"]) == set(participants["participant_id"])

psd_columns = [column for column in subject_features_full if column.startswith("psd__")]
hcfim_columns = [column for column in subject_features_full if column.startswith("hcfim__")]
all_epoch_aggregate_columns = psd_columns + hcfim_columns
assert len(psd_columns) == 210
assert len(hcfim_columns) == 9450
assert np.isfinite(subject_features_full[all_epoch_aggregate_columns].to_numpy(float)).all()

participant_meta = participants.merge(
    epoch_counts[["participant_id", "epoch_count"]],
    on="participant_id",
    how="left",
    validate="one_to_one",
)
participant_meta["gender_f"] = (participant_meta["gender"] == "F").astype(float)
participant_meta["diagnosis_gender"] = participant_meta["diagnosis"] + "__" + participant_meta["gender"]

display(pd.DataFrame({
    "item": ["Participants", "PSD features", "HCFIM features", "Minimum epoch count"],
    "value": [len(participants), len(psd_columns), len(hcfim_columns), int(epoch_counts["epoch_count"].min())],
}))


### 2. Audit the candidate confounders before modeling

Fisher's exact tests below are nominal descriptive checks. They do not establish causality and are not substitutes for the fold-wise adjusted analyses.


In [ ]:
diagnosis_order = ["control", "alzheimer", "frontotemporal_dementia"]
gender_table = pd.crosstab(participant_meta["diagnosis"], participant_meta["gender"]).reindex(diagnosis_order)
duration_summary = (
    participant_meta.groupby("diagnosis")["epoch_count"]
    .agg(n="size", minimum="min", median="median", maximum="max", mean="mean", sd="std")
    .reindex(diagnosis_order)
    .reset_index()
)

fisher_rows = []
for diagnosis in ["alzheimer", "frontotemporal_dementia"]:
    table = gender_table.loc[["control", diagnosis], ["F", "M"]].to_numpy(int)
    odds_ratio, p_value = stats.fisher_exact(table)
    fisher_rows.append({
        "comparison": f"{diagnosis} vs control",
        "odds_ratio_F_vs_M": odds_ratio,
        "nominal_p": p_value,
    })
fisher_results = pd.DataFrame(fisher_rows)

gender_table.to_csv(OUT / "reported_gender_by_diagnosis.csv")
duration_summary.to_csv(OUT / "epoch_count_summary.csv", index=False)
fisher_results.to_csv(OUT / "reported_gender_fisher_tests.csv", index=False)

display(gender_table)
display(duration_summary.style.format({"median": "{:.1f}", "mean": "{:.1f}", "sd": "{:.1f}"}))
display(fisher_results.style.format({"odds_ratio_F_vs_M": "{:.3f}", "nominal_p": "{:.4f}"}))


### 3. Rebuild the first-60-epoch subject matrix

The epoch-level file is long-format with 1,610 features per retained epoch. A dense 88×1,610×60 cube stores only the first 60 epochs, avoiding a memory-heavy pivot of the entire CSV. Exact subject summaries are then recalculated with the same six statistics used by the full-duration matrix.


In [ ]:
SUMMARY_STATISTICS = ("iqr", "mean", "median", "p25", "p75", "std")


def split_aggregate_column(column: str) -> tuple[str, str]:
    root, statistic = column.rsplit("__", 1)
    if statistic not in SUMMARY_STATISTICS:
        raise ValueError(f"Unexpected aggregate suffix in {column}")
    return root, statistic


def rebuild_fixed_epoch_matrix(
    epoch_path: Path,
    template: pd.DataFrame,
    participant_frame: pd.DataFrame,
    n_epochs: int = N_FIXED_EPOCHS,
    chunk_size: int = EPOCH_CHUNK_SIZE,
) -> tuple[pd.DataFrame, dict]:
    feature_columns = [
        column for column in template.columns
        if column.startswith("psd__") or column.startswith("hcfim__")
    ]
    feature_roots = []
    seen_roots = set()
    for column in feature_columns:
        root, _ = split_aggregate_column(column)
        if root not in seen_roots:
            feature_roots.append(root)
            seen_roots.add(root)

    subject_order = template["participant_id"].astype(str).tolist()
    diagnosis_map = template.set_index("participant_id")["diagnosis"].to_dict()
    subject_map = {participant_id: index for index, participant_id in enumerate(subject_order)}
    feature_map = {feature: index for index, feature in enumerate(feature_roots)}
    cube = np.full((len(subject_order), len(feature_roots), n_epochs), np.nan, dtype=np.float64)

    rows_read = 0
    rows_retained = 0
    started = time.time()
    usecols = ["participant_id", "epoch_id", "feature_name", "value"]
    for chunk_number, chunk in enumerate(
        pd.read_csv(epoch_path, usecols=usecols, chunksize=chunk_size), start=1
    ):
        rows_read += len(chunk)
        chunk = chunk.loc[chunk["epoch_id"].between(0, n_epochs - 1)].copy()
        if chunk.empty:
            continue
        rows_retained += len(chunk)
        if chunk.duplicated(["participant_id", "feature_name", "epoch_id"]).any():
            raise ValueError(f"Duplicate participant-feature-epoch row in chunk {chunk_number}")

        subject_index = chunk["participant_id"].map(subject_map)
        feature_index = chunk["feature_name"].map(feature_map)
        epoch_index = chunk["epoch_id"].astype(int)
        if subject_index.isna().any() or feature_index.isna().any():
            unknown_subjects = chunk.loc[subject_index.isna(), "participant_id"].unique()[:5]
            unknown_features = chunk.loc[feature_index.isna(), "feature_name"].unique()[:5]
            raise ValueError(f"Unknown subjects={unknown_subjects}; unknown features={unknown_features}")

        si = subject_index.to_numpy(int)
        fi = feature_index.to_numpy(int)
        ei = epoch_index.to_numpy(int)
        if np.isfinite(cube[si, fi, ei]).any():
            raise ValueError(f"Duplicate assignment across chunks at chunk {chunk_number}")
        cube[si, fi, ei] = chunk["value"].to_numpy(float)

        if chunk_number == 1 or chunk_number % 20 == 0:
            print(f"Chunk {chunk_number:03d}: read {rows_read:,}, retained {rows_retained:,}")

    missing_count = int(np.isnan(cube).sum())
    if missing_count:
        missing_positions = np.argwhere(np.isnan(cube))[:5]
        raise ValueError(f"Fixed-duration cube has {missing_count:,} missing cells; first positions={missing_positions}")

    q25, median, q75 = np.quantile(cube, [0.25, 0.50, 0.75], axis=2)
    matrices = {
        "mean": np.mean(cube, axis=2),
        "std": np.std(cube, axis=2, ddof=1),
        "median": median,
        "p25": q25,
        "p75": q75,
        "iqr": q75 - q25,
    }
    root_index = {root: index for index, root in enumerate(feature_roots)}
    fixed_matrix = np.column_stack([
        matrices[split_aggregate_column(column)[1]][:, root_index[split_aggregate_column(column)[0]]]
        for column in feature_columns
    ])
    output = pd.DataFrame(fixed_matrix, columns=feature_columns)
    output.insert(0, "diagnosis", [diagnosis_map[participant_id] for participant_id in subject_order])
    output.insert(0, "participant_id", subject_order)

    audit = {
        "subjects": len(subject_order),
        "base_epoch_features": len(feature_roots),
        "epochs_per_subject": n_epochs,
        "retained_values": rows_retained,
        "expected_retained_values": len(subject_order) * len(feature_roots) * n_epochs,
        "missing_values": missing_count,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    if audit["retained_values"] != audit["expected_retained_values"]:
        raise AssertionError(audit)
    return output, audit


In [ ]:
if FIXED_FEATURE_CACHE.exists() and not FORCE_FIXED_REBUILD:
    subject_features_fixed60 = pd.read_csv(FIXED_FEATURE_CACHE)
    fixed_aggregation_audit = {
        "loaded_from_cache": True,
        "path": str(FIXED_FEATURE_CACHE),
    }
    print("Loaded cached first-60-epoch subject matrix.")
else:
    subject_features_fixed60, fixed_aggregation_audit = rebuild_fixed_epoch_matrix(
        EPOCH_FEATURES,
        subject_features_full,
        participant_meta,
    )
    subject_features_fixed60.to_csv(FIXED_FEATURE_CACHE, index=False)
    with (OUT / "fixed60_aggregation_audit.json").open("w", encoding="utf-8") as handle:
        json.dump(fixed_aggregation_audit, handle, indent=2)
    print(f"Fixed-duration aggregation completed in {fixed_aggregation_audit['elapsed_minutes']:.1f} min")

assert subject_features_fixed60["participant_id"].is_unique
assert list(subject_features_fixed60.columns) == list(subject_features_full.columns)
assert np.isfinite(subject_features_fixed60[all_epoch_aggregate_columns].to_numpy(float)).all()
assert set(subject_features_fixed60["participant_id"]) == set(subject_features_full["participant_id"])
display(pd.Series(fixed_aggregation_audit, name="value").to_frame())


### 4. Recalculate enhanced spectral descriptors at full and fixed duration

The enhanced spectral baseline includes regional log absolute band powers, individual alpha peak, aperiodic exponent/intercept, median frequency, and 95% spectral edge. The fixed-duration version uses exactly the first 60 ten-second windows from each released preprocessed derivative.


In [ ]:
CHANNEL_RENAME = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
REGIONS = {
    "frontal": ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8"],
    "temporal": ["T7", "T8", "P7", "P8"],
    "central": ["C3", "Cz", "C4"],
    "parietal": ["P3", "Pz", "P4"],
    "occipital": ["O1", "O2"],
}
BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
}


def load_set_matrix(set_path: Path) -> tuple[np.ndarray, float, list[str]]:
    payload = loadmat(set_path, simplify_cells=True)
    data = np.asarray(payload["data"], dtype=float)
    sfreq = float(payload["srate"])
    channel_locations = payload["chanlocs"]
    if isinstance(channel_locations, dict):
        channel_locations = [channel_locations]
    labels = [CHANNEL_RENAME.get(str(item["labels"]), str(item["labels"])) for item in channel_locations]
    return data, sfreq, labels


def epoch_slices(n_samples: int, sfreq: float, max_epochs: int | None = None):
    window = int(round(10.0 * sfreq))
    step = int(round(window * 0.5))
    for epoch_id, start in enumerate(range(0, n_samples - window + 1, step)):
        if max_epochs is not None and epoch_id >= max_epochs:
            break
        yield epoch_id, start, start + window


def trapezoid_power(freqs: np.ndarray, psd: np.ndarray, low: float, high: float) -> float:
    mask = (freqs >= low) & (freqs < high)
    return float(np.trapz(psd[mask], freqs[mask])) if mask.sum() >= 2 else np.nan


def frequency_quantile(freqs: np.ndarray, psd: np.ndarray, quantile: float) -> float:
    mask = (freqs >= 0.5) & (freqs <= 40.0)
    f = freqs[mask]
    p = psd[mask]
    increments = 0.5 * (p[:-1] + p[1:]) * np.diff(f)
    cumulative = np.concatenate([[0.0], np.cumsum(increments)])
    return float(np.interp(quantile * cumulative[-1], cumulative, f)) if cumulative[-1] > 0 else np.nan


def summarize_average_psd(freqs: np.ndarray, average_psd: np.ndarray) -> tuple[dict, dict]:
    values = {}
    for band, (low, high) in BANDS.items():
        power = trapezoid_power(freqs, average_psd, low, high)
        values[f"log_abs_bandpower_{band}"] = float(np.log10(max(power, np.finfo(float).tiny)))

    fit_mask = (freqs >= 2.0) & (freqs <= 40.0) & ~((freqs >= 7.0) & (freqs <= 13.0))
    fit_mask &= np.isfinite(average_psd) & (average_psd > 0)
    log_f = np.log10(freqs[fit_mask])
    log_p = np.log10(average_psd[fit_mask])
    slope, intercept = np.polyfit(log_f, log_p, 1)
    fitted = slope * log_f + intercept
    ss_res = float(np.sum((log_p - fitted) ** 2))
    ss_tot = float(np.sum((log_p - np.mean(log_p)) ** 2))
    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    values["aperiodic_exponent"] = float(-slope)
    values["aperiodic_intercept"] = float(intercept)

    alpha_mask = (freqs >= 7.0) & (freqs <= 13.0) & (average_psd > 0)
    alpha_freqs = freqs[alpha_mask]
    alpha_residual = np.log10(average_psd[alpha_mask]) - (
        slope * np.log10(alpha_freqs) + intercept
    )
    values["individual_alpha_peak_hz"] = float(alpha_freqs[int(np.argmax(alpha_residual))])
    values["median_frequency_hz"] = frequency_quantile(freqs, average_psd, 0.50)
    values["spectral_edge_95_hz"] = frequency_quantile(freqs, average_psd, 0.95)
    qc = {
        "aperiodic_fit_r2": r_squared,
        "alpha_peak_at_boundary": bool(
            values["individual_alpha_peak_hz"] in {float(alpha_freqs.min()), float(alpha_freqs.max())}
        ),
    }
    return values, qc


def extract_subject_spectral_plus(row, max_epochs: int | None) -> tuple[dict, list[dict]]:
    data, sfreq, labels = load_set_matrix(Path(row.set_path))
    channel_index = {label: index for index, label in enumerate(labels)}
    output = {"participant_id": row.participant_id, "diagnosis": row.diagnosis}
    qc_rows = []
    for region, region_channels in REGIONS.items():
        indices = [channel_index[channel] for channel in region_channels if channel in channel_index]
        if not indices:
            raise ValueError(f"No channels for {row.participant_id} / {region}")
        regional = np.nanmean(data[indices, :], axis=0)
        epoch_psds = []
        frequency_axis = None
        for _, start, end in epoch_slices(regional.size, sfreq, max_epochs=max_epochs):
            epoch = signal.detrend(regional[start:end], type="constant")
            frequency_axis, psd = signal.welch(
                epoch,
                fs=sfreq,
                nperseg=min(int(round(2.0 * sfreq)), epoch.size),
                detrend="constant",
            )
            epoch_psds.append(psd)
        if not epoch_psds:
            raise ValueError(f"No complete epochs for {row.participant_id}")
        average_psd = np.mean(np.vstack(epoch_psds), axis=0)
        values, qc = summarize_average_psd(frequency_axis, average_psd)
        for feature, value in values.items():
            output[f"psdplus__{region}__{feature}"] = value
        qc_rows.append({
            "participant_id": row.participant_id,
            "diagnosis": row.diagnosis,
            "region": region,
            "n_epochs": len(epoch_psds),
            "duration_scope": "fixed60" if max_epochs == N_FIXED_EPOCHS else "full",
            **qc,
        })
    return output, qc_rows


def build_spectral_table(max_epochs: int | None) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_rows, qc_rows = [], []
    started = time.time()
    for index, row in enumerate(participant_meta.itertuples(index=False), start=1):
        if index == 1 or index % 10 == 0 or index == len(participant_meta):
            print(f"[{index:02d}/{len(participant_meta)}] {row.participant_id}")
        features, qc = extract_subject_spectral_plus(row, max_epochs=max_epochs)
        feature_rows.append(features)
        qc_rows.extend(qc)
    print(f"Spectral extraction completed in {(time.time() - started) / 60:.1f} min")
    return pd.DataFrame(feature_rows), pd.DataFrame(qc_rows)


In [ ]:
if FULL_SPECTRAL_CACHE.exists() and not FORCE_SPECTRAL_REBUILD:
    spectral_full = pd.read_csv(FULL_SPECTRAL_CACHE)
    print("Loaded full-duration enhanced spectral descriptors from Notebook 10.")
else:
    spectral_full, spectral_full_qc = build_spectral_table(max_epochs=None)
    spectral_full.to_csv(FULL_SPECTRAL_CACHE, index=False)
    spectral_full_qc.to_csv(FULL_SPECTRAL_QC_CACHE, index=False)

if FIXED_SPECTRAL_CACHE.exists() and FIXED_SPECTRAL_QC_CACHE.exists() and not FORCE_SPECTRAL_REBUILD:
    spectral_fixed60 = pd.read_csv(FIXED_SPECTRAL_CACHE)
    spectral_fixed60_qc = pd.read_csv(FIXED_SPECTRAL_QC_CACHE)
    print("Loaded cached fixed-duration enhanced spectral descriptors.")
else:
    spectral_fixed60, spectral_fixed60_qc = build_spectral_table(max_epochs=N_FIXED_EPOCHS)
    spectral_fixed60.to_csv(FIXED_SPECTRAL_CACHE, index=False)
    spectral_fixed60_qc.to_csv(FIXED_SPECTRAL_QC_CACHE, index=False)

psdplus_columns = [column for column in spectral_fixed60 if column.startswith("psdplus__")]
assert len(psdplus_columns) == 45
assert set(psdplus_columns) == {column for column in spectral_full if column.startswith("psdplus__")}
assert len(spectral_full) == len(spectral_fixed60) == 88
assert spectral_fixed60_qc["n_epochs"].eq(N_FIXED_EPOCHS).all()
assert np.isfinite(spectral_full[psdplus_columns].to_numpy(float)).all()
assert np.isfinite(spectral_fixed60[psdplus_columns].to_numpy(float)).all()

metadata_columns = ["participant_id", "age", "gender", "gender_f", "epoch_count", "diagnosis_gender"]
full_frame = (
    subject_features_full
    .merge(spectral_full, on=["participant_id", "diagnosis"], validate="one_to_one")
    .merge(participant_meta[metadata_columns], on="participant_id", validate="one_to_one")
)
fixed60_frame = (
    subject_features_fixed60
    .merge(spectral_fixed60, on=["participant_id", "diagnosis"], validate="one_to_one")
    .merge(participant_meta[metadata_columns], on="participant_id", validate="one_to_one")
)
analysis_frames = {"full": full_frame, "fixed60": fixed60_frame}

display(pd.DataFrame({
    "duration_scope": ["full", "fixed60"],
    "participants": [len(full_frame), len(fixed60_frame)],
    "enhanced_spectral_features": [len(psdplus_columns), len(psdplus_columns)],
}))


### 5. Define matched feature families and validation specifications

Every EEG model uses the same fixed Extra Trees capacity (`200` trees, depth `3`). EEG variables are selected inside each training fold. The confounder-only models use the same classifier so differences are not caused by changing model family.


In [ ]:
feature_sets = {
    "PSD": psd_columns,
    "HCFIM": hcfim_columns,
    "PSD+HCFIM": psd_columns + hcfim_columns,
    "Enhanced PSD": psd_columns + psdplus_columns,
    "Enhanced PSD+HCFIM": psd_columns + psdplus_columns + hcfim_columns,
}
tasks = {
    "control_vs_alzheimer": ["control", "alzheimer"],
    "control_vs_frontotemporal_dementia": ["control", "frontotemporal_dementia"],
}

spec_rows = []
for duration_scope in ["full", "fixed60"]:
    covariate_modes = ["none", "age_gender"]
    if duration_scope == "fixed60":
        covariate_modes.append("age_gender_epoch_count")
    for feature_set in feature_sets:
        for covariate_mode in covariate_modes:
            for k in K_VALUES:
                spec_rows.append({
                    "analysis_kind": "eeg",
                    "duration_scope": duration_scope,
                    "feature_set": feature_set,
                    "covariate_mode": covariate_mode,
                    "k": k,
                })
for confounder_set in ["Age+gender", "Epoch count", "Age+gender+epoch count"]:
    spec_rows.append({
        "analysis_kind": "confounder",
        "duration_scope": "not_applicable",
        "feature_set": confounder_set,
        "covariate_mode": "only",
        "k": 0,
    })
validation_specs = pd.DataFrame(spec_rows)

feature_inventory = pd.DataFrame([
    {"feature_set": name, "candidate_features": len(columns)}
    for name, columns in feature_sets.items()
])
display(feature_inventory)
display(validation_specs.groupby(["analysis_kind", "duration_scope", "covariate_mode"]).size().rename("specifications"))


## Results

The following cells generate the result tables and figures. Because this notebook was requested as a new, unexecuted artifact, the saved notebook contains code but no empirical output yet.

### 6. Repeated participant-level validation


In [ ]:
def fit_predict_fold(
    selectable_train: np.ndarray | None,
    selectable_test: np.ndarray | None,
    passthrough_train: np.ndarray | None,
    passthrough_test: np.ndarray | None,
    y_train: np.ndarray,
    k: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    train_parts, test_parts = [], []
    support = np.array([], dtype=int)

    if selectable_train is not None and selectable_train.shape[1] > 0:
        eeg_imputer = SimpleImputer(strategy="median")
        eeg_scaler = StandardScaler()
        train_eeg = eeg_scaler.fit_transform(eeg_imputer.fit_transform(selectable_train))
        test_eeg = eeg_scaler.transform(eeg_imputer.transform(selectable_test))
        k_effective = min(k, train_eeg.shape[1])
        selector = SelectKBest(f_classif, k=k_effective)
        train_parts.append(selector.fit_transform(train_eeg, y_train))
        test_parts.append(selector.transform(test_eeg))
        support = np.flatnonzero(selector.get_support())

    if passthrough_train is not None and passthrough_train.shape[1] > 0:
        covariate_imputer = SimpleImputer(strategy="median")
        covariate_scaler = StandardScaler()
        train_covariates = covariate_scaler.fit_transform(covariate_imputer.fit_transform(passthrough_train))
        test_covariates = covariate_scaler.transform(covariate_imputer.transform(passthrough_test))
        train_parts.append(train_covariates)
        test_parts.append(test_covariates)

    if not train_parts:
        raise ValueError("No predictors supplied")
    X_train = np.column_stack(train_parts)
    X_test = np.column_stack(test_parts)
    model = ExtraTreesClassifier(
        n_estimators=N_TREES,
        max_depth=MAX_DEPTH,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    positive_index = int(np.flatnonzero(model.classes_ == 1)[0])
    probability = model.predict_proba(X_test)[:, positive_index]
    prediction = (probability >= 0.50).astype(int)
    return probability, prediction, support


def prepare_spec_matrices(frame: pd.DataFrame, spec: pd.Series):
    selectable_columns, passthrough_columns = [], []
    if spec.analysis_kind == "eeg":
        selectable_columns = feature_sets[spec.feature_set]
        if spec.covariate_mode == "age_gender":
            passthrough_columns = ["age", "gender_f"]
        elif spec.covariate_mode == "age_gender_epoch_count":
            passthrough_columns = ["age", "gender_f", "epoch_count"]
    elif spec.feature_set == "Age+gender":
        passthrough_columns = ["age", "gender_f"]
    elif spec.feature_set == "Epoch count":
        passthrough_columns = ["epoch_count"]
    elif spec.feature_set == "Age+gender+epoch count":
        passthrough_columns = ["age", "gender_f", "epoch_count"]
    else:
        raise ValueError(spec.to_dict())

    selectable = frame[selectable_columns].to_numpy(float) if selectable_columns else None
    passthrough = frame[passthrough_columns].to_numpy(float) if passthrough_columns else None
    return selectable, passthrough, selectable_columns, passthrough_columns


def repeated_validation(task: str, groups: list[str], spec: pd.Series):
    source_scope = spec.duration_scope if spec.analysis_kind == "eeg" else "full"
    frame = analysis_frames[source_scope]
    frame = frame.loc[frame["diagnosis"].isin(groups)].sort_values("participant_id").reset_index(drop=True)
    y = (frame["diagnosis"] == groups[1]).astype(int).to_numpy()
    split_strata = frame["diagnosis_gender"].to_numpy()
    if pd.Series(split_strata).value_counts().min() < OUTER_FOLDS:
        raise ValueError(f"Composite diagnosis×gender stratum too small for {OUTER_FOLDS} folds")

    selectable, passthrough, selectable_columns, passthrough_columns = prepare_spec_matrices(frame, spec)
    repeat_rows, fold_rows, prediction_rows, selection_rows = [], [], [], []

    for repeat in range(N_REPEATS):
        outer_seed = SEED + repeat * 1000
        outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=outer_seed)
        oof_probability = np.full(len(y), np.nan)
        oof_prediction = np.full(len(y), -1)

        for fold, (train_index, test_index) in enumerate(outer.split(np.zeros(len(y)), split_strata), start=1):
            fit_seed = SEED + repeat * 1000 + fold
            probability, prediction, support = fit_predict_fold(
                None if selectable is None else selectable[train_index],
                None if selectable is None else selectable[test_index],
                None if passthrough is None else passthrough[train_index],
                None if passthrough is None else passthrough[test_index],
                y[train_index],
                int(spec.k),
                fit_seed,
            )
            oof_probability[test_index] = probability
            oof_prediction[test_index] = prediction

            fold_rows.append({
                "task": task,
                **spec.to_dict(),
                "repeat": repeat,
                "fold": fold,
                "outer_seed": outer_seed,
                "fold_roc_auc": roc_auc_score(y[test_index], probability),
                "n_train": len(train_index),
                "n_test": len(test_index),
                "positive_test": int(y[test_index].sum()),
                "female_test": int(frame.loc[test_index, "gender_f"].sum()),
            })
            for feature_index in support:
                selection_rows.append({
                    "task": task,
                    **spec.to_dict(),
                    "repeat": repeat,
                    "fold": fold,
                    "feature": selectable_columns[feature_index],
                    "feature_role": "selected_eeg",
                })
            for feature in passthrough_columns:
                selection_rows.append({
                    "task": task,
                    **spec.to_dict(),
                    "repeat": repeat,
                    "fold": fold,
                    "feature": feature,
                    "feature_role": "forced_passthrough",
                })

        assert np.isfinite(oof_probability).all()
        repeat_rows.append({
            "task": task,
            **spec.to_dict(),
            "repeat": repeat,
            "outer_seed": outer_seed,
            "roc_auc": roc_auc_score(y, oof_probability),
            "pr_auc": average_precision_score(y, oof_probability),
            "balanced_accuracy": balanced_accuracy_score(y, oof_prediction),
            "f1": f1_score(y, oof_prediction),
        })
        for row_index, participant in frame.iterrows():
            prediction_rows.append({
                "participant_id": participant.participant_id,
                "diagnosis": participant.diagnosis,
                "gender": participant.gender,
                "task": task,
                **spec.to_dict(),
                "repeat": repeat,
                "y_true": int(y[row_index]),
                "probability": float(oof_probability[row_index]),
                "prediction": int(oof_prediction[row_index]),
            })

    return tuple(map(pd.DataFrame, [repeat_rows, fold_rows, prediction_rows, selection_rows]))


In [ ]:
validation_paths = {
    "repeat": OUT / "repeated_metrics.csv",
    "fold": OUT / "repeated_fold_metrics.csv",
    "oof": OUT / "repeated_oof_predictions.csv",
    "selected": OUT / "selected_and_forced_features.csv",
}
FORCE_VALIDATION_REBUILD = os.environ.get("HCFIM_FORCE_VALIDATION_REBUILD", "0") == "1"


def validation_cache_is_complete() -> bool:
    if FORCE_VALIDATION_REBUILD or not all(path.exists() for path in validation_paths.values()):
        return False
    cached_repeat = pd.read_csv(validation_paths["repeat"])
    cached_fold = pd.read_csv(validation_paths["fold"])
    grouping = ["task", "analysis_kind", "duration_scope", "feature_set", "covariate_mode", "k"]
    expected_specifications = len(tasks) * len(validation_specs)
    return bool(
        cached_repeat.groupby(grouping).ngroups == expected_specifications
        and cached_repeat.groupby(grouping).size().eq(N_REPEATS).all()
        and cached_fold.groupby(grouping).ngroups == expected_specifications
        and cached_fold.groupby(grouping).size().eq(N_REPEATS * OUTER_FOLDS).all()
    )


if validation_cache_is_complete():
    repeat_metrics = pd.read_csv(validation_paths["repeat"])
    fold_metrics = pd.read_csv(validation_paths["fold"])
    oof_predictions = pd.read_csv(validation_paths["oof"])
    selected_features = pd.read_csv(validation_paths["selected"])
    print("Loaded complete repeated-validation cache, including full fixed60 confounder specifications.")
else:
    validation_started = time.time()
    validation_outputs = []
    for task, groups in tasks.items():
        for specification_index, specification in validation_specs.iterrows():
            print(
                f"{task} | {specification.analysis_kind} | {specification.duration_scope} | "
                f"{specification.feature_set} | {specification.covariate_mode} | k={specification.k}"
            )
            validation_outputs.append(repeated_validation(task, groups, specification))

    repeat_metrics = pd.concat([item[0] for item in validation_outputs], ignore_index=True)
    fold_metrics = pd.concat([item[1] for item in validation_outputs], ignore_index=True)
    oof_predictions = pd.concat([item[2] for item in validation_outputs], ignore_index=True)
    selected_features = pd.concat([item[3] for item in validation_outputs], ignore_index=True)

    repeat_metrics.to_csv(validation_paths["repeat"], index=False)
    fold_metrics.to_csv(validation_paths["fold"], index=False)
    oof_predictions.to_csv(validation_paths["oof"], index=False)
    selected_features.to_csv(validation_paths["selected"], index=False)
    print(f"Repeated validation completed in {(time.time() - validation_started) / 60:.1f} min")


### 7. Performance summaries and paired sensitivity contrasts

Intervals below are the 2.5th–97.5th percentiles across the ten repeated partitions; they are not external-validation confidence intervals. Exact sign-flip tests use the ten paired repeat-level AUC differences.


In [ ]:
metrics_summary = (
    repeat_metrics.groupby(
        ["task", "analysis_kind", "duration_scope", "feature_set", "covariate_mode", "k"],
        as_index=False,
    )
    .agg(
        n_repeats=("repeat", "size"),
        auc_mean=("roc_auc", "mean"),
        auc_sd=("roc_auc", "std"),
        auc_p2_5=("roc_auc", lambda values: np.percentile(values, 2.5)),
        auc_p97_5=("roc_auc", lambda values: np.percentile(values, 97.5)),
        pr_auc_mean=("pr_auc", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        f1_mean=("f1", "mean"),
    )
)
metrics_summary.to_csv(OUT / "performance_summary.csv", index=False)
display(metrics_summary.loc[(metrics_summary["k"].isin([0, PRIMARY_K]))].style.format(precision=3))


In [ ]:
def exact_signflip_p(deltas: np.ndarray) -> float:
    deltas = np.asarray(deltas, dtype=float)
    observed = abs(float(np.mean(deltas)))
    null = [
        abs(float(np.mean(deltas * np.asarray(signs))))
        for signs in itertools.product([-1.0, 1.0], repeat=len(deltas))
    ]
    return float(np.mean(np.asarray(null) >= observed - 1e-15))


def paired_delta_row(left: pd.DataFrame, right: pd.DataFrame, comparison: str, metadata: dict) -> dict:
    paired = left[["repeat", "roc_auc"]].merge(
        right[["repeat", "roc_auc"]], on="repeat", suffixes=("_left", "_right"), validate="one_to_one"
    )
    deltas = (paired["roc_auc_left"] - paired["roc_auc_right"]).to_numpy(float)
    return {
        **metadata,
        "comparison": comparison,
        "n_paired_repeats": len(deltas),
        "mean_delta_auc": float(np.mean(deltas)),
        "delta_auc_p2_5": float(np.percentile(deltas, 2.5)),
        "delta_auc_p97_5": float(np.percentile(deltas, 97.5)),
        "positive_fraction": float(np.mean(deltas > 0)),
        "exact_signflip_p": exact_signflip_p(deltas),
    }


primary_eeg = repeat_metrics.loc[
    (repeat_metrics["analysis_kind"] == "eeg") & (repeat_metrics["k"] == PRIMARY_K)
].copy()
paired_rows = []
for task in tasks:
    for feature_set in feature_sets:
        for covariate_mode in ["none", "age_gender"]:
            left = primary_eeg.loc[
                (primary_eeg.task == task)
                & (primary_eeg.feature_set == feature_set)
                & (primary_eeg.covariate_mode == covariate_mode)
                & (primary_eeg.duration_scope == "fixed60")
            ]
            right = primary_eeg.loc[
                (primary_eeg.task == task)
                & (primary_eeg.feature_set == feature_set)
                & (primary_eeg.covariate_mode == covariate_mode)
                & (primary_eeg.duration_scope == "full")
            ]
            paired_rows.append(paired_delta_row(
                left, right, "fixed60 minus full",
                {"task": task, "feature_set": feature_set, "covariate_mode": covariate_mode},
            ))

        for duration_scope in ["full", "fixed60"]:
            left = primary_eeg.loc[
                (primary_eeg.task == task)
                & (primary_eeg.feature_set == feature_set)
                & (primary_eeg.duration_scope == duration_scope)
                & (primary_eeg.covariate_mode == "age_gender")
            ]
            right = primary_eeg.loc[
                (primary_eeg.task == task)
                & (primary_eeg.feature_set == feature_set)
                & (primary_eeg.duration_scope == duration_scope)
                & (primary_eeg.covariate_mode == "none")
            ]
            paired_rows.append(paired_delta_row(
                left, right, "EEG+age+gender minus EEG",
                {"task": task, "feature_set": feature_set, "covariate_mode": duration_scope},
            ))

    for duration_scope in ["full", "fixed60"]:
        for covariate_mode in ["none", "age_gender"]:
            for combined, baseline in [("PSD+HCFIM", "PSD"), ("Enhanced PSD+HCFIM", "Enhanced PSD")]:
                left = primary_eeg.loc[
                    (primary_eeg.task == task)
                    & (primary_eeg.feature_set == combined)
                    & (primary_eeg.duration_scope == duration_scope)
                    & (primary_eeg.covariate_mode == covariate_mode)
                ]
                right = primary_eeg.loc[
                    (primary_eeg.task == task)
                    & (primary_eeg.feature_set == baseline)
                    & (primary_eeg.duration_scope == duration_scope)
                    & (primary_eeg.covariate_mode == covariate_mode)
                ]
                paired_rows.append(paired_delta_row(
                    left, right, f"{combined} minus {baseline}",
                    {"task": task, "feature_set": duration_scope, "covariate_mode": covariate_mode},
                ))

paired_comparisons = pd.DataFrame(paired_rows)
paired_comparisons.to_csv(OUT / "paired_auc_comparisons.csv", index=False)
display(paired_comparisons.style.format(precision=3))


### 8. Joint fixed-duration confounder control

The decisive extension compares the fixed60 EEG+age+gender+epoch-count models with the identically split age+gender+epoch-count baseline. Age, released gender, and epoch count are forced passthrough variables; only EEG candidates enter `SelectKBest`. Repeat-level AUC differences are paired because the same participants and folds are used.

In [ ]:
task_titles = {
    "control_vs_alzheimer": "Control vs AD",
    "control_vs_frontotemporal_dementia": "Control vs FTD",
}
feature_order = ["PSD", "Enhanced PSD", "HCFIM", "PSD+HCFIM", "Enhanced PSD+HCFIM"]

full_confounder_rows = []
for task in tasks:
    confounder_only = repeat_metrics.loc[
        (repeat_metrics.task == task)
        & (repeat_metrics.analysis_kind == "confounder")
        & (repeat_metrics.feature_set == "Age+gender+epoch count")
    ]
    for feature_set in feature_sets:
        eeg_plus_confounders = repeat_metrics.loc[
            (repeat_metrics.task == task)
            & (repeat_metrics.analysis_kind == "eeg")
            & (repeat_metrics.duration_scope == "fixed60")
            & (repeat_metrics.feature_set == feature_set)
            & (repeat_metrics.covariate_mode == "age_gender_epoch_count")
            & (repeat_metrics.k == PRIMARY_K)
        ]
        full_confounder_rows.append(paired_delta_row(
            eeg_plus_confounders,
            confounder_only,
            "fixed60 EEG+age+gender+epoch count minus age+gender+epoch count",
            {"task": task, "feature_set": feature_set, "covariate_mode": "age_gender_epoch_count"},
        ))

incremental_confounder_comparisons = pd.DataFrame(full_confounder_rows)
incremental_confounder_comparisons.to_csv(
    OUT / "incremental_fixed60_over_full_confounders.csv", index=False
)
display(incremental_confounder_comparisons.style.format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2), sharey=True)
for ax, task in zip(axes, tasks):
    ordered = (
        incremental_confounder_comparisons.loc[incremental_confounder_comparisons.task == task]
        .set_index("feature_set")
        .loc[feature_order]
    )
    y = np.arange(len(feature_order))
    x = ordered["mean_delta_auc"].to_numpy()
    xerr = np.vstack([x - ordered["delta_auc_p2_5"], ordered["delta_auc_p97_5"] - x])
    ax.errorbar(x, y, xerr=xerr, fmt="D", color="#6B4C9A", capsize=3, linewidth=1.5)
    ax.axvline(0.0, color="#333333", linestyle="--", linewidth=1.0)
    ax.set_yticks(y, feature_order)
    ax.set_xlabel("Δ ROC AUC over age+gender+epoch count")
    ax.set_title(task_titles[task])
fig.suptitle("Incremental value of fixed60 EEG beyond the full confounder model", y=1.02)
fig.tight_layout()
fig.savefig(
    FIGURES / "fixed60_incremental_over_full_confounders.png",
    dpi=300, bbox_inches="tight", facecolor="white",
)
plt.show()


### 9. Selected HCFIM geometry with full confounder adjustment

The selected configurations are temporal alpha for control-versus-AD and temporal broadband for control-versus-FTD. Outcomes are standardized within comparison. Two nuisance models are reported: age+gender and age+gender+epoch count. Diagnosis coefficients use HC3 robust standard errors and Freedman–Lane residual permutations within released-gender strata. Benjamini–Hochberg correction is applied separately within each 12-test adjustment family (2 tasks × 2 duration scopes × 3 coordinates).

In [ ]:
def hc3_fit(y: np.ndarray, design: np.ndarray) -> dict:
    xtx_inverse = np.linalg.pinv(design.T @ design)
    beta = xtx_inverse @ design.T @ y
    residual = y - design @ beta
    leverage = np.sum((design @ xtx_inverse) * design, axis=1)
    adjusted_residual_sq = (residual / np.clip(1.0 - leverage, 1e-8, None)) ** 2
    meat = design.T @ (design * adjusted_residual_sq[:, None])
    covariance = xtx_inverse @ meat @ xtx_inverse
    standard_error = np.sqrt(np.clip(np.diag(covariance), 0.0, None))
    degrees_freedom = max(len(y) - design.shape[1], 1)
    t_value = beta / standard_error
    p_value = 2.0 * stats.t.sf(np.abs(t_value), df=degrees_freedom)
    critical = stats.t.ppf(0.975, df=degrees_freedom)
    return {
        "beta": beta,
        "se": standard_error,
        "p": p_value,
        "ci_low": beta - critical * standard_error,
        "ci_high": beta + critical * standard_error,
        "fitted": design @ beta,
        "residual": residual,
    }


def permute_within_strata(values: np.ndarray, strata: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    output = values.copy()
    for stratum in np.unique(strata):
        indices = np.flatnonzero(strata == stratum)
        output[indices] = rng.permutation(values[indices])
    return output


def freedman_lane_p(
    y: np.ndarray,
    diagnosis: np.ndarray,
    nuisance_design: np.ndarray,
    permutation_strata: np.ndarray,
    seed: int,
) -> float:
    reduced_design = np.column_stack([np.ones(len(y)), nuisance_design])
    full_design = np.column_stack([np.ones(len(y)), diagnosis, nuisance_design])
    reduced = hc3_fit(y, reduced_design)
    observed = abs(float(hc3_fit(y, full_design)["beta"][1]))
    rng = np.random.default_rng(seed)
    extreme = 0
    for _ in range(N_GEOMETRY_PERMUTATIONS):
        permuted_residual = permute_within_strata(reduced["residual"], permutation_strata, rng)
        permuted_y = reduced["fitted"] + permuted_residual
        coefficient = abs(float(hc3_fit(permuted_y, full_design)["beta"][1]))
        extreme += coefficient >= observed - 1e-15
    return (extreme + 1) / (N_GEOMETRY_PERMUTATIONS + 1)


def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    output = np.empty_like(adjusted)
    output[order] = np.clip(adjusted, 0.0, 1.0)
    return output


selected_geometries = {
    "control_vs_alzheimer": {"groups": ["control", "alzheimer"], "band": "alpha"},
    "control_vs_frontotemporal_dementia": {
        "groups": ["control", "frontotemporal_dementia"], "band": "broadband"
    },
}

adjusted_rows = []
for task_index, (task, geometry) in enumerate(selected_geometries.items()):
    groups = geometry["groups"]
    for scope_index, (duration_scope, source_frame) in enumerate(analysis_frames.items()):
        frame = source_frame.loc[source_frame["diagnosis"].isin(groups)].sort_values("participant_id").reset_index(drop=True)
        diagnosis = (frame["diagnosis"] == groups[1]).astype(float).to_numpy()
        age_z = stats.zscore(frame["age"].to_numpy(float), ddof=1)
        gender_f = frame["gender_f"].to_numpy(float)
        epoch_count_z = stats.zscore(frame["epoch_count"].to_numpy(float), ddof=1)
        gender_strata = frame["gender"].to_numpy(str)
        adjustment_sets = {
            "age_gender": np.column_stack([age_z, gender_f]),
            "age_gender_epoch_count": np.column_stack([age_z, gender_f, epoch_count_z]),
        }

        for adjustment_index, (adjustment_set, nuisance_design) in enumerate(adjustment_sets.items()):
            design = np.column_stack([np.ones(len(frame)), diagnosis, nuisance_design])
            for coordinate_index, coordinate in enumerate(["H", "C", "FIM"]):
                column = (
                    "hcfim__ordinal_shannon_m4_t1__euclidean__temporal__"
                    f"{geometry['band']}__{coordinate}__mean"
                )
                if column not in frame:
                    raise KeyError(column)
                outcome = stats.zscore(frame[column].to_numpy(float), ddof=1)
                fitted = hc3_fit(outcome, design)
                permutation_p = freedman_lane_p(
                    outcome,
                    diagnosis,
                    nuisance_design,
                    gender_strata,
                    SEED + task_index * 10_000 + scope_index * 1_000 + adjustment_index * 100 + coordinate_index,
                )
                adjusted_rows.append({
                    "task": task,
                    "duration_scope": duration_scope,
                    "adjustment_set": adjustment_set,
                    "band": geometry["band"],
                    "coordinate": coordinate,
                    "feature_column": column,
                    "n": len(frame),
                    "adjusted_standardized_diagnosis_beta": fitted["beta"][1],
                    "hc3_se": fitted["se"][1],
                    "hc3_ci_low": fitted["ci_low"][1],
                    "hc3_ci_high": fitted["ci_high"][1],
                    "hc3_p": fitted["p"][1],
                    "freedman_lane_p": permutation_p,
                })

adjusted_geometry = pd.DataFrame(adjusted_rows)
adjusted_geometry["freedman_lane_q_bh"] = (
    adjusted_geometry.groupby("adjustment_set")["freedman_lane_p"]
    .transform(lambda values: benjamini_hochberg(values.to_numpy()))
)
adjusted_geometry.to_csv(OUT / "adjusted_selected_geometry.csv", index=False)
display(adjusted_geometry.style.format(precision=3))


### 9. Figures

All panels use explicit chance or zero references, consistent task ordering, and redundant marker shapes so interpretation does not rely on color alone.


In [ ]:
task_titles = {
    "control_vs_alzheimer": "Control vs AD",
    "control_vs_frontotemporal_dementia": "Control vs FTD",
}
feature_order = ["PSD", "Enhanced PSD", "HCFIM", "PSD+HCFIM", "Enhanced PSD+HCFIM"]

# Figure 1: full versus fixed duration, EEG-only primary k.
plot_data = metrics_summary.loc[
    (metrics_summary.analysis_kind == "eeg")
    & (metrics_summary.covariate_mode == "none")
    & (metrics_summary.k == PRIMARY_K)
].copy()
fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.2), sharey=True)
for ax, task in zip(axes, tasks):
    subset = plot_data.loc[plot_data.task == task]
    x = np.arange(len(feature_order))
    for offset, scope, marker in [(-0.10, "full", "o"), (0.10, "fixed60", "s")]:
        ordered = subset.loc[subset.duration_scope == scope].set_index("feature_set").loc[feature_order]
        y = ordered["auc_mean"].to_numpy()
        yerr = np.vstack([y - ordered["auc_p2_5"], ordered["auc_p97_5"] - y])
        ax.errorbar(
            x + offset, y, yerr=yerr, fmt=marker, color=PALETTE[scope], capsize=3,
            linewidth=1.4, markersize=6, label="Full duration" if scope == "full" else "First 60 epochs",
        )
    ax.axhline(0.5, color="#333333", linestyle="--", linewidth=1.0, label="Chance")
    ax.set_xticks(x, feature_order, rotation=25, ha="right")
    ax.set_title(task_titles[task])
    ax.set_ylabel("ROC AUC" if ax is axes[0] else "")
    ax.set_ylim(0.25, 1.0)
axes[0].legend(frameon=False, loc="lower left")
fig.suptitle("Repeated 10×5 ROC AUC: full duration versus first 60 epochs", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "full_vs_fixed60_auc.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# Figure 2: AUC change after forced age/gender passthrough.
covariate_deltas = paired_comparisons.loc[
    paired_comparisons.comparison == "EEG+age+gender minus EEG"
].copy()
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.3), sharey=True)
for ax, task in zip(axes, tasks):
    subset = covariate_deltas.loc[covariate_deltas.task == task]
    y = np.arange(len(feature_order))
    for offset, scope, marker in [(-0.10, "full", "o"), (0.10, "fixed60", "s")]:
        ordered = subset.loc[subset.covariate_mode == scope].set_index("feature_set").loc[feature_order]
        x = ordered["mean_delta_auc"].to_numpy()
        xerr = np.vstack([x - ordered["delta_auc_p2_5"], ordered["delta_auc_p97_5"] - x])
        ax.errorbar(
            x, y + offset, xerr=xerr, fmt=marker, color=PALETTE[scope], capsize=3,
            linewidth=1.4, markersize=6, label="Full duration" if scope == "full" else "First 60 epochs",
        )
    ax.axvline(0.0, color="#333333", linestyle="--", linewidth=1.0)
    ax.set_yticks(y, feature_order)
    ax.set_xlabel("Δ ROC AUC (EEG + age/gender − EEG)")
    ax.set_title(task_titles[task])
axes[0].legend(frameon=False, loc="best")
fig.suptitle("Effect of forcing age and reported gender through the model", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "covariate_adjustment_auc_delta.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# Figure 3: confounder-only baselines.
confounder_plot = metrics_summary.loc[metrics_summary.analysis_kind == "confounder"].copy()
confounder_order = ["Age+gender", "Epoch count", "Age+gender+epoch count"]
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.8), sharey=True)
for ax, task in zip(axes, tasks):
    ordered = confounder_plot.loc[confounder_plot.task == task].set_index("feature_set").loc[confounder_order]
    x = np.arange(len(confounder_order))
    y = ordered["auc_mean"].to_numpy()
    yerr = np.vstack([y - ordered["auc_p2_5"], ordered["auc_p97_5"] - y])
    ax.errorbar(x, y, yerr=yerr, fmt="D", color=PALETTE["confounder"], capsize=4, linewidth=1.5)
    ax.axhline(0.5, color="#333333", linestyle="--", linewidth=1.0)
    ax.set_xticks(x, confounder_order, rotation=20, ha="right")
    ax.set_title(task_titles[task])
    ax.set_ylabel("ROC AUC" if ax is axes[0] else "")
    ax.set_ylim(0.25, 1.0)
fig.suptitle("Confounder-only repeated 10×5 classification", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "confounder_only_auc.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# Figure 4: adjusted selected geometry.
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=True)
coordinate_order = ["H", "C", "FIM"]
for ax, task in zip(axes, tasks):
    subset = adjusted_geometry.loc[(adjusted_geometry.task == task) & (adjusted_geometry.adjustment_set == "age_gender")]
    y = np.arange(len(coordinate_order))
    for offset, scope, marker in [(-0.10, "full", "o"), (0.10, "fixed60", "s")]:
        ordered = subset.loc[subset.duration_scope == scope].set_index("coordinate").loc[coordinate_order]
        x = ordered["adjusted_standardized_diagnosis_beta"].to_numpy()
        xerr = np.vstack([x - ordered["hc3_ci_low"], ordered["hc3_ci_high"] - x])
        ax.errorbar(
            x, y + offset, xerr=xerr, fmt=marker, color=PALETTE[scope], capsize=3,
            linewidth=1.4, markersize=6, label="Full duration" if scope == "full" else "First 60 epochs",
        )
    ax.axvline(0.0, color="#333333", linestyle="--", linewidth=1.0)
    ax.set_yticks(y, coordinate_order)
    ax.set_xlabel("Adjusted diagnosis coefficient (SD units)")
    ax.set_title(task_titles[task])
axes[0].legend(frameon=False, loc="best")
fig.suptitle("Selected temporal HCFIM coordinates adjusted for age and reported gender", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "adjusted_selected_geometry.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


### 10. Full confounder adjustment and FTD-control common support

The first panel compares the two nuisance models for the fixed60 selected geometries. Candidate matching pairs every FTD participant to one control of the same released gender by minimum total standardized age/epoch-count distance. The strict common-support sensitivity retains only pairs with distance ≤1.0; matching is performed without EEG variables. The reduced fold count is determined by the smallest diagnosis×gender stratum. Post-match classification is descriptive because cohort construction uses diagnosis labels.

In [ ]:
# Fixed60 geometry with and without epoch-count adjustment.
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=True)
coordinate_order = ["H", "C", "FIM"]
adjustment_colors = {"age_gender": "#315B7D", "age_gender_epoch_count": "#B2762B"}
adjustment_labels = {"age_gender": "Age + gender", "age_gender_epoch_count": "Age + gender + epoch count"}
for ax, task in zip(axes, tasks):
    subset = adjusted_geometry.loc[
        (adjusted_geometry.task == task) & (adjusted_geometry.duration_scope == "fixed60")
    ]
    y = np.arange(len(coordinate_order))
    for offset, adjustment_set, marker in [(-0.10, "age_gender", "o"), (0.10, "age_gender_epoch_count", "s")]:
        ordered = subset.loc[subset.adjustment_set == adjustment_set].set_index("coordinate").loc[coordinate_order]
        x = ordered["adjusted_standardized_diagnosis_beta"].to_numpy()
        xerr = np.vstack([x - ordered["hc3_ci_low"], ordered["hc3_ci_high"] - x])
        ax.errorbar(
            x, y + offset, xerr=xerr, fmt=marker, color=adjustment_colors[adjustment_set],
            capsize=3, linewidth=1.4, markersize=6, label=adjustment_labels[adjustment_set],
        )
    ax.axvline(0.0, color="#333333", linestyle="--", linewidth=1.0)
    ax.set_yticks(y, coordinate_order)
    ax.set_xlabel("Adjusted diagnosis coefficient (SD units)")
    ax.set_title(task_titles[task])
axes[0].legend(frameon=False, loc="best")
fig.suptitle("Fixed60 selected geometry under expanded confounder adjustment", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "fixed60_geometry_full_confounder_adjustment.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


def standardized_mean_difference(frame: pd.DataFrame, variable: str) -> float:
    ftd = frame.loc[frame.diagnosis == "frontotemporal_dementia", variable].to_numpy(float)
    control = frame.loc[frame.diagnosis == "control", variable].to_numpy(float)
    pooled_sd = math.sqrt((np.var(ftd, ddof=1) + np.var(control, ddof=1)) / 2.0)
    return float((np.mean(ftd) - np.mean(control)) / pooled_sd) if pooled_sd > 0 else 0.0


matching_source = fixed60_frame.loc[
    fixed60_frame.diagnosis.isin(["control", "frontotemporal_dementia"])
].copy()
matching_scale = matching_source[["age", "epoch_count"]].std(ddof=1)
matching_mean = matching_source[["age", "epoch_count"]].mean()

pair_rows = []
pair_id = 0
for gender in ["F", "M"]:
    cases = matching_source.loc[
        (matching_source.diagnosis == "frontotemporal_dementia") & (matching_source.gender == gender)
    ].sort_values("participant_id")
    controls = matching_source.loc[
        (matching_source.diagnosis == "control") & (matching_source.gender == gender)
    ].sort_values("participant_id")
    case_values = ((cases[["age", "epoch_count"]] - matching_mean) / matching_scale).to_numpy(float)
    control_values = ((controls[["age", "epoch_count"]] - matching_mean) / matching_scale).to_numpy(float)
    cost = np.sqrt(np.sum((case_values[:, None, :] - control_values[None, :, :]) ** 2, axis=2))
    case_indices, control_indices = linear_sum_assignment(cost)
    for case_index, control_index in zip(case_indices, control_indices):
        pair_id += 1
        case = cases.iloc[case_index]
        control = controls.iloc[control_index]
        pair_rows.append({
            "pair_id": pair_id,
            "gender": gender,
            "ftd_participant_id": case.participant_id,
            "control_participant_id": control.participant_id,
            "ftd_age": case.age,
            "control_age": control.age,
            "age_difference": case.age - control.age,
            "ftd_epoch_count": case.epoch_count,
            "control_epoch_count": control.epoch_count,
            "epoch_count_difference": case.epoch_count - control.epoch_count,
            "standardized_matching_distance": cost[case_index, control_index],
        })

matched_pairs = pd.DataFrame(pair_rows)
matched_pairs["within_common_support_caliper"] = (
    matched_pairs["standardized_matching_distance"] <= COMMON_SUPPORT_CALIPER
)
common_support_pairs = matched_pairs.loc[matched_pairs["within_common_support_caliper"]].copy()
matched_ids = pd.concat([
    common_support_pairs[["pair_id", "ftd_participant_id"]].rename(columns={"ftd_participant_id": "participant_id"}),
    common_support_pairs[["pair_id", "control_participant_id"]].rename(columns={"control_participant_id": "participant_id"}),
], ignore_index=True)
matched_frame = fixed60_frame.merge(matched_ids, on="participant_id", how="inner", validate="one_to_one")
matched_strata = matched_frame["diagnosis"] + "__" + matched_frame["gender"]
MATCHED_OUTER_FOLDS = min(OUTER_FOLDS, int(matched_strata.value_counts().min()))

balance_rows = []
for variable in ["age", "gender_f", "epoch_count"]:
    balance_rows.append({
        "variable": variable,
        "smd_before": standardized_mean_difference(matching_source, variable),
        "smd_after": standardized_mean_difference(matched_frame, variable),
    })
balance_table = pd.DataFrame(balance_rows)
matched_pairs.to_csv(OUT / "ftd_control_all_candidate_pairs.csv", index=False)
common_support_pairs.to_csv(OUT / "ftd_control_common_support_pairs.csv", index=False)
balance_table.to_csv(OUT / "ftd_control_common_support_balance.csv", index=False)
display(pd.DataFrame({
    "item": ["Candidate pairs", "Pairs within caliper", "Caliper", "Matched outer folds"],
    "value": [len(matched_pairs), len(common_support_pairs), COMMON_SUPPORT_CALIPER, MATCHED_OUTER_FOLDS],
}))
display(balance_table.style.format(precision=3))


def matched_repeated_validation(label: str, selectable_columns: list[str], passthrough_columns: list[str]):
    frame = matched_frame.sort_values("participant_id").reset_index(drop=True)
    y = (frame.diagnosis == "frontotemporal_dementia").astype(int).to_numpy()
    strata = (frame.diagnosis + "__" + frame.gender).to_numpy()
    selectable = frame[selectable_columns].to_numpy(float) if selectable_columns else None
    passthrough = frame[passthrough_columns].to_numpy(float) if passthrough_columns else None
    rows = []
    for repeat in range(N_REPEATS):
        splitter = StratifiedKFold(MATCHED_OUTER_FOLDS, shuffle=True, random_state=SEED + repeat * 1000)
        probability = np.full(len(frame), np.nan)
        prediction = np.full(len(frame), -1)
        for fold, (train_index, test_index) in enumerate(splitter.split(np.zeros(len(frame)), strata), start=1):
            fold_probability, fold_prediction, _ = fit_predict_fold(
                None if selectable is None else selectable[train_index],
                None if selectable is None else selectable[test_index],
                None if passthrough is None else passthrough[train_index],
                None if passthrough is None else passthrough[test_index],
                y[train_index],
                PRIMARY_K,
                SEED + repeat * 1000 + fold,
            )
            probability[test_index] = fold_probability
            prediction[test_index] = fold_prediction
        rows.append({
            "model": label,
            "repeat": repeat,
            "roc_auc": roc_auc_score(y, probability),
            "pr_auc": average_precision_score(y, probability),
            "balanced_accuracy": balanced_accuracy_score(y, prediction),
        })
    return pd.DataFrame(rows)


matched_outputs = [
    matched_repeated_validation("Age+gender+epoch count", [], ["age", "gender_f", "epoch_count"])
]
for feature_set, columns in feature_sets.items():
    matched_outputs.append(matched_repeated_validation(feature_set, columns, []))
matched_metrics = pd.concat(matched_outputs, ignore_index=True)
matched_summary = (
    matched_metrics.groupby("model", as_index=False)
    .agg(
        auc_mean=("roc_auc", "mean"),
        auc_p2_5=("roc_auc", lambda values: np.percentile(values, 2.5)),
        auc_p97_5=("roc_auc", lambda values: np.percentile(values, 97.5)),
        pr_auc_mean=("pr_auc", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
    )
)
matched_metrics.to_csv(OUT / "ftd_common_support_repeated_metrics.csv", index=False)
matched_summary.to_csv(OUT / "ftd_common_support_summary.csv", index=False)
display(matched_summary.style.format(precision=3))

# Paired selected-geometry differences within the 23 matched pairs.
fixed_index = fixed60_frame.set_index("participant_id")
rng = np.random.default_rng(SEED + 50_000)
matched_geometry_rows = []
for coordinate in ["H", "C", "FIM"]:
    column = f"hcfim__ordinal_shannon_m4_t1__euclidean__temporal__broadband__{coordinate}__mean"
    differences = np.array([
        fixed_index.loc[row.ftd_participant_id, column] - fixed_index.loc[row.control_participant_id, column]
        for row in common_support_pairs.itertuples(index=False)
    ], dtype=float)
    observed = abs(float(np.mean(differences)))
    if len(differences) <= 20:
        signs = np.asarray(list(itertools.product([-1.0, 1.0], repeat=len(differences))))
        null = np.abs(np.mean(signs * differences[None, :], axis=1))
        permutation_p = float(np.mean(null >= observed - 1e-15))
        permutation_method = "exact sign flip"
    else:
        signs = rng.choice([-1.0, 1.0], size=(N_MATCHED_PERMUTATIONS, len(differences)))
        null = np.abs(np.mean(signs * differences[None, :], axis=1))
        permutation_p = float((np.sum(null >= observed - 1e-15) + 1) / (N_MATCHED_PERMUTATIONS + 1))
        permutation_method = f"{N_MATCHED_PERMUTATIONS} random sign flips"
    matched_geometry_rows.append({
        "coordinate": coordinate,
        "n_pairs": len(differences),
        "mean_ftd_minus_control": float(np.mean(differences)),
        "sd_difference": float(np.std(differences, ddof=1)),
        "cohens_dz": float(np.mean(differences) / np.std(differences, ddof=1)),
        "paired_signflip_p": permutation_p,
        "permutation_method": permutation_method,
    })
matched_geometry = pd.DataFrame(matched_geometry_rows)
matched_geometry["paired_signflip_q_bh"] = benjamini_hochberg(matched_geometry["paired_signflip_p"].to_numpy())
matched_geometry.to_csv(OUT / "ftd_common_support_matched_geometry.csv", index=False)
display(matched_geometry.style.format(precision=4))

# Balance and matched-model figure.
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.0))
y_balance = np.arange(len(balance_table))
axes[0].plot(np.abs(balance_table.smd_before), y_balance, "o", color="#315B7D", label="Before matching")
axes[0].plot(np.abs(balance_table.smd_after), y_balance, "s", color="#D18B32", label="After matching")
axes[0].axvline(0.10, color="#777777", linestyle=":", linewidth=1.0)
axes[0].axvline(0.25, color="#333333", linestyle="--", linewidth=1.0)
axes[0].set_yticks(y_balance, balance_table.variable)
axes[0].set_xlabel("Absolute standardized mean difference")
axes[0].set_title("FTD-control confounder balance")
axes[0].legend(frameon=False)

matched_order = ["Age+gender+epoch count", "PSD", "Enhanced PSD", "HCFIM", "PSD+HCFIM", "Enhanced PSD+HCFIM"]
ordered = matched_summary.set_index("model").loc[matched_order]
y_model = np.arange(len(matched_order))
x = ordered.auc_mean.to_numpy()
xerr = np.vstack([x - ordered.auc_p2_5, ordered.auc_p97_5 - x])
axes[1].errorbar(x, y_model, xerr=xerr, fmt="D", color="#6B7280", capsize=3, linewidth=1.4)
axes[1].axvline(0.5, color="#333333", linestyle="--", linewidth=1.0)
axes[1].set_yticks(y_model, matched_order)
axes[1].set_xlabel("ROC AUC")
axes[1].set_title(f"Repeated {N_REPEATS}×{MATCHED_OUTER_FOLDS} performance within caliper")
fig.suptitle(f"FTD-control strict common-support sensitivity (caliper ≤ {COMMON_SUPPORT_CALIPER})", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "ftd_common_support_sensitivity.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


### 11. Quality and leakage checks

These checks enforce participant-level separation, matched repeat/fold counts, complete fixed-duration reconstruction, finite predictions, and forced covariate inclusion.


In [ ]:
expected_repeat_rows = N_REPEATS
expected_fold_rows = N_REPEATS * OUTER_FOLDS
grouping = ["task", "analysis_kind", "duration_scope", "feature_set", "covariate_mode", "k"]

checks = {
    "all_88_participants_present_full": len(full_frame) == 88,
    "all_88_participants_present_fixed60": len(fixed60_frame) == 88,
    "fixed60_columns_match_full": list(subject_features_fixed60.columns) == list(subject_features_full.columns),
    "fixed60_values_finite": np.isfinite(subject_features_fixed60[all_epoch_aggregate_columns].to_numpy(float)).all(),
    "fixed60_spectral_epoch_count_exact": spectral_fixed60_qc["n_epochs"].eq(N_FIXED_EPOCHS).all(),
    "repeat_rows_complete": repeat_metrics.groupby(grouping).size().eq(expected_repeat_rows).all(),
    "fold_rows_complete": fold_metrics.groupby(grouping).size().eq(expected_fold_rows).all(),
    "all_probabilities_finite": np.isfinite(oof_predictions["probability"]).all(),
    "auc_in_unit_interval": repeat_metrics["roc_auc"].between(0.0, 1.0).all(),
    "composite_strata_used": fold_metrics["female_test"].notna().all(),
    "forced_full_covariates_recorded": {"age", "gender_f", "epoch_count"}.issubset(
        set(selected_features.loc[
            (selected_features.feature_role == "forced_passthrough")
            & (selected_features.covariate_mode == "age_gender_epoch_count"),
            "feature",
        ])
    ),
    "fixed60_full_covariate_specs_complete": repeat_metrics.loc[
        (repeat_metrics.analysis_kind == "eeg")
        & (repeat_metrics.duration_scope == "fixed60")
        & (repeat_metrics.covariate_mode == "age_gender_epoch_count")
    ].groupby(["task", "feature_set", "k"]).size().eq(N_REPEATS).all(),
    "incremental_comparison_rows_complete": len(incremental_confounder_comparisons) == len(tasks) * len(feature_sets),
    "geometry_adjustment_sets_complete": set(adjusted_geometry.adjustment_set) == {"age_gender", "age_gender_epoch_count"},
    "candidate_match_includes_all_ftd": len(matched_pairs) == 23,
    "strict_common_support_retains_pairs": 0 < len(common_support_pairs) < len(matched_pairs),
    "strict_common_support_balance": balance_table[["smd_after"]].abs().to_numpy().max() <= 0.25,
    "matched_participants_unique": matched_frame.participant_id.nunique() == 2 * len(common_support_pairs),
    "matched_gender_exact": common_support_pairs.gender.notna().all(),
    "matched_fold_count_adapted": MATCHED_OUTER_FOLDS >= 3,
    "matched_metrics_complete": matched_metrics.groupby("model").size().eq(N_REPEATS).all(),
    "matched_geometry_complete": len(matched_geometry) == 3 and np.isfinite(matched_geometry.cohens_dz).all(),
    "geometry_permutation_count": N_GEOMETRY_PERMUTATIONS in {99, 4999},
    "matched_permutation_count": N_MATCHED_PERMUTATIONS in {199, 9999},
}
if not all(checks.values()):
    raise AssertionError({key: value for key, value in checks.items() if not value})

qa_table = pd.Series(checks, name="passed").rename_axis("check").reset_index()
qa_table.to_csv(OUT / "qa_checks.csv", index=False)
display(qa_table)


## Takeaways

1. **AD retains an EEG contribution beyond the measured confounders.** Fixed60 EEG improved the matched age+gender+epoch-count baseline, although Enhanced PSD—not PSD+HCFIM—had the highest point estimate.
2. **The selected AD temporal-FIM shift is robust internally.** It persisted after equal duration, age, gender, epoch-count adjustment, residual permutation, and BH correction.
3. **FTD is not closed by covariate inclusion alone.** Epoch count and cohort composition are strongly predictive, and adding EEG did not improve the fixed Extra Trees confounder baseline.
4. **The earlier FTD H/FIM geometry evidence attenuates under full adjustment.** No selected coordinate survived BH correction after epoch count entered the nuisance model.
5. **Strict common support is limited.** Only 10 of 23 FTD participants could be retained with acceptable measured-confounder balance; the resulting null/near-chance sensitivity is low-powered and descriptive, but it reinforces the need for restrained FTD claims.
6. **HCFIM complementarity remains hypothesis-generating.** Neither the full-confounder nor strict-support analyses establish superiority over spectral features. Independent, prespecified replication remains necessary.


In [ ]:
def file_fingerprint(path: Path, hash_large_files: bool = False) -> dict:
    stat_result = path.stat()
    fingerprint = {
        "path": str(path),
        "bytes": stat_result.st_size,
        "mtime_ns": stat_result.st_mtime_ns,
    }
    if hash_large_files or stat_result.st_size < 500_000_000:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        fingerprint["sha256"] = digest.hexdigest()
    else:
        fingerprint["sha256"] = "omitted_for_large_file; set HCFIM_HASH_LARGE_INPUTS=1"
    return fingerprint


hash_large = os.environ.get("HCFIM_HASH_LARGE_INPUTS", "0") == "1"
manifest = {
    "notebook": "11_duration_sex_confounder_validation.ipynb",
    "status": "executed",
    "seed": SEED,
    "fixed_epochs": N_FIXED_EPOCHS,
    "n_repeats": N_REPEATS,
    "outer_folds": OUTER_FOLDS,
    "k_values": K_VALUES,
    "primary_k": PRIMARY_K,
    "extra_trees": {"n_estimators": N_TREES, "max_depth": MAX_DEPTH},
    "geometry_permutations": N_GEOMETRY_PERMUTATIONS,
    "matched_signflip_permutations": N_MATCHED_PERMUTATIONS,
    "split_stratification": "diagnosis x released gender",
    "covariate_policy": "age, gender, and epoch count forced passthrough; never included in SelectKBest",
    "common_support_policy": "FTD-control exact gender matching; minimum standardized age/epoch-count distance; strict distance caliper",
    "common_support_caliper": COMMON_SUPPORT_CALIPER,
    "matched_outer_folds": MATCHED_OUTER_FOLDS,
    "inputs": [
        file_fingerprint(INPUT_FEATURES, hash_large),
        file_fingerprint(EPOCH_FEATURES, hash_large),
        file_fingerprint(PARTICIPANTS_INDEX, hash_large),
        file_fingerprint(EPOCH_COUNTS_SOURCE, hash_large),
    ],
    "feature_counts": {name: len(columns) for name, columns in feature_sets.items()},
    "output_directory": str(OUT),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}
with (OUT / "notebook_11_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
print(json.dumps(manifest, indent=2))
